# Spectroscopic reference set (part 2)

Real S-PLUS galaxies whose SFH labels come from **Bagpipes fits to their SDSS spectra under exactly the training prior** (`synthetic_model.py`), so they test the ML model on real data with labels it can match. Firefly (SDSS DR16) is an independent check of those fits, never a label.

**Inputs:** `dr3_corrected.csv`, `../Data/sdss_eboss_firefly-dr16.fits`, `../dustMaps/csfd/`, `training_set.parquet` (part 1), SDSS spectra (downloaded), S-PLUS Cloud aperture photometry (needs `SPLUS_USERNAME` / `SPLUS_PASSWORD`).
**Outputs:** `spectra/`, `spectral_fits/jobs/` (one `.npz` per fit job), `spectral_fits/` (one `.h5` per finished fit, plus `fit_log.csv` and, for Slurm tasks, `fit_logs/<job_id>.csv`), `spectral_fits/selection.csv` (selected galaxies, append-only), `spectral_fits/failures.csv` (download and prepare failures), `spectral_fits/calibration_mocks.joblib`, `spectroscopic_reference.csv`, `firefly_comparison.png`.
**Run:** top to bottom with the *Inference4ETGs (MultiNest)* kernel, after `training_set.ipynb`. The notebook does not run the fits: it exports one job per galaxy and per calibration mock to `spectral_fits/jobs/`, and `run_spectral_fits.py` fits them outside the notebook (one core per fit, on a many-core PC or a Slurm cluster; set `RUN_FITS_HERE = True` to start it from the notebook). Copy the resulting `.h5` files back into `spectral_fits/` and re-run the analysis cells: they accept zero, some or all fits. The fitter must pass a **calibration test** on mocks drawn from the prior; the verdict is stored as `fitter_calibrated` in the reference CSV.

Spectra fit 3″ (SDSS) or 2″ (BOSS) fibres, so photometry is the 3″ S-PLUS aperture, and BOSS galaxies are reported separately. The **test** half of the split is used once, for the final ML evaluation; with a few tens of galaxies this is a pilot-size set.

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy.io import fits
from astroquery.sdss import SDSS
from dustmaps.csfd import CSFDQuery
import joblib

import synthetic_model as sm

# Files and folders
SPLUS_FILE = "dr3_corrected.csv"
FIREFLY_FILE = "../Data/sdss_eboss_firefly-dr16.fits"
DUST_MAP_FILE = "../dustMaps/csfd/csfd_ebv.fits"
DUST_MASK_FILE = "../dustMaps/csfd/mask.fits"
TRAINING_FILE = "training_set.parquet"
SPECTRA_DIRECTORY = "spectra"
FITS_DIRECTORY = "spectral_fits"
JOBS_DIRECTORY = "spectral_fits/jobs"                   # exported fit jobs, one .npz each
SUPERSEDED_DIRECTORY = "spectral_fits/superseded"       # results of earlier, replaced runs
FAILURES_FILE = "spectral_fits/failures.csv"
SELECTION_FILE = "spectral_fits/selection.csv"   # selected specobjids, append-only across passes
MOCK_FILE = "spectral_fits/calibration_mocks.joblib"   # calibration mocks and true labels, kept so their fits stay checkable
OLD_MOCK_FILE = "spectral_fits/mocks.joblib"           # the first run's inject-and-recover mocks (superseded)
OUTPUT_FILE = "spectroscopic_reference.csv"
FIGURE_FILE = "firefly_comparison.png"
CREDENTIALS_FILE = "../.env"     # git-ignored; holds SPLUS_USERNAME and SPLUS_PASSWORD
APERTURE_CACHE_FILE = "splus_aperture_photometry.csv"   # raw 3" photometry of every S-PLUS x Firefly match

# Sample
MIN_PROB_GALAXY = 0.8
MATCH_RADIUS = 1.5               # [arcsec]
N_REDSHIFT_BINS = 4
N_AGE_BINS = 3
GALAXIES_PER_CELL = 5
N_REAL_GALAXIES = N_REDSHIFT_BINS * N_AGE_BINS * GALAXIES_PER_CELL   # 60
FIBER_DIAMETER = {"26": 3.0, "v5_13_0": 2.0}   # [arcsec] SDSS legacy and BOSS
SQL_CHUNK = 200                  # specobjids per SDSS SQL query

# Fits (run outside the notebook by run_spectral_fits.py; sm.N_CALIBRATION_MOCKS, sm.MAX_ITERATIONS and
# sm.MIN_COVERAGE_95 set the calibration test and the iteration cap)
RUN_FITS_HERE = False            # True: run_spectral_fits.py is started from the notebook, on N_WORKERS cores
N_WORKERS = 4                    # [cores] used when RUN_FITS_HERE is True; fits take hours, so set it for your machine
LIBMULTINEST_DIRECTORY = "../.venv/lib"   # folder with libmultinest, put on LD_LIBRARY_PATH for the fits
SEED = 0

# S-PLUS aperture photometry. DR4 keeps one table per band, each with its own RA/Dec columns
SPLUS_RA_COLUMN = "RA"
SPLUS_DEC_COLUMN = "DEC"
APERTURE_SOURCES = {}
for band in sm.BANDS:
    APERTURE_SOURCES[band] = (f"dr4_dual.dr4_dual_{band.lower()}", f"{band}_aper_3", f"e_{band}_aper_3")
SPLUS_QUERY_TIMEOUT = 600        # [s]
SPLUS_EXECUTION_MODE = "sync"    # small queries; async jobs are capped at 4 concurrent per account
SPLUS_QUERY_ATTEMPTS = 3         # tries per S-PLUS query before giving up (count)
SPLUS_RETRY_WAIT = 30            # [s] wait after the first failed try; doubles after each further one

SSFR_PASSIVE = -11.0             # [log yr^-1] star-forming above, passive below

os.makedirs(SPECTRA_DIRECTORY, exist_ok=True)
os.makedirs(FITS_DIRECTORY, exist_ok=True)
failures = []

## 1. Sample
S-PLUS DR3 galaxies matched to Firefly DR16 spectra within 1.5″, with spectroscopic redshift inside the prior. One spectrum per S-PLUS galaxy (the closest match).

In [ ]:
# 1. S-PLUS galaxies
splus = pd.read_csv(SPLUS_FILE)
splus = splus[splus["PROB_GAL"] > MIN_PROB_GALAXY].reset_index(drop=True)

# 2. Firefly galaxies with a redshift inside the prior, in the S-PLUS declination range
FIREFLY_COLUMNS = ["SPECOBJID", "RUN2D", "PLATE", "MJD", "FIBERID", "RA", "DEC", "Z_NOQSO", "CLASS_NOQSO",
                   "Chabrier_MILES_age_massW", "Chabrier_MILES_metallicity_massW", "Chabrier_MILES_total_mass"]
firefly_data = fits.getdata(FIREFLY_FILE, 1)
firefly = {}
for column in FIREFLY_COLUMNS:
    values = np.asarray(firefly_data[column])
    # FITS stores big-endian numbers; pandas needs the machine's native byte order
    firefly[column] = values.astype(values.dtype.newbyteorder("="))
firefly = pd.DataFrame(firefly)
firefly["CLASS_NOQSO"] = firefly["CLASS_NOQSO"].str.strip()
firefly["RUN2D"] = firefly["RUN2D"].str.strip()
is_galaxy = firefly["CLASS_NOQSO"] == "GALAXY"
in_prior = (firefly["Z_NOQSO"] > sm.Z_RANGE[0]) & (firefly["Z_NOQSO"] < sm.Z_RANGE[1])
in_footprint = (firefly["DEC"] > splus["DEC"].min() - 1) & (firefly["DEC"] < splus["DEC"].max() + 1)
firefly = firefly[is_galaxy & in_prior & in_footprint].reset_index(drop=True)
del firefly_data

# 3. Nearest S-PLUS galaxy of each spectrum
spectrum_coordinates = SkyCoord(firefly["RA"].to_numpy() * u.deg, firefly["DEC"].to_numpy() * u.deg)
splus_coordinates = SkyCoord(splus["RA"].to_numpy() * u.deg, splus["DEC"].to_numpy() * u.deg)
nearest_splus, separation, _ = match_coordinates_sky(spectrum_coordinates, splus_coordinates)
firefly["splus_index"] = nearest_splus
firefly["separation"] = separation.arcsec
matched = firefly[firefly["separation"] < MATCH_RADIUS]

# 4. One spectrum per S-PLUS galaxy: the closest
matched = matched.sort_values("separation").drop_duplicates("splus_index").reset_index(drop=True)
# Kept as text: eBOSS specobjids exceed the int64 range
matched["specobjid"] = matched["SPECOBJID"].str.strip()
matched["fiber_diameter"] = matched["RUN2D"].map(FIBER_DIAMETER)
for column in ("Chabrier_MILES_age_massW", "Chabrier_MILES_metallicity_massW", "Chabrier_MILES_total_mass"):
    missing = matched[column] <= sm.FIREFLY_MISSING
    matched.loc[missing, column] = np.nan
print(f"{len(matched)} S-PLUS galaxies with an SDSS/BOSS spectrum; "
      f"{matched['fiber_diameter'].eq(3.0).sum()} SDSS, {matched['fiber_diameter'].eq(2.0).sum()} BOSS")

In [ ]:
# 1. Median S/N of each spectrum from the SDSS database
sn_by_id = {}
identifiers = matched["specobjid"].tolist()
for start in range(0, len(identifiers), SQL_CHUNK):
    chunk = identifiers[start:start + SQL_CHUNK]
    id_list = ", ".join(chunk)
    query = ("SELECT CAST(specObjID AS VARCHAR(20)) AS specobjid_text, snMedian "
             f"FROM SpecObjAll WHERE specObjID IN ({id_list})")
    result = SDSS.query_sql(query, data_release=16)
    for row in result:
        sn_by_id[str(row["specobjid_text"]).strip()] = float(row["snMedian"])
matched["sn_median"] = matched["specobjid"].map(sn_by_id)

# 2. Grid of redshift x Firefly age quantile bins; highest S/N first in each cell
candidates = matched.dropna(subset=["Chabrier_MILES_age_massW", "sn_median"]).copy()
candidates["redshift_bin"] = pd.qcut(candidates["Z_NOQSO"], N_REDSHIFT_BINS, labels=False)
candidates["age_bin"] = pd.qcut(candidates["Chabrier_MILES_age_massW"], N_AGE_BINS, labels=False)

chosen = []
for (redshift_bin, age_bin), cell in candidates.groupby(["redshift_bin", "age_bin"]):
    best = cell.sort_values("sn_median", ascending=False).head(GALAXIES_PER_CELL).copy()
    best["rank_in_cell"] = np.arange(len(best))
    chosen.append(best)
# Round-robin order (best of every cell first), so a budget cut never empties a whole cell
picks = pd.concat(chosen, ignore_index=True)
picks = picks.sort_values(["rank_in_cell", "redshift_bin", "age_bin"]).reset_index(drop=True)
print(f"{len(picks)} galaxies picked this pass (target {N_REAL_GALAXIES})")
display(pd.crosstab(picks["redshift_bin"], picks["age_bin"]))

# 3. Selection kept across passes. The stored ids stay, in their stored order (so the first wave and the
# split never change), and this pass's new picks are appended: a later pass can only add galaxies
stored_identifiers = []
if os.path.exists(SELECTION_FILE):
    stored_identifiers = pd.read_csv(SELECTION_FILE, dtype=str)["specobjid"].tolist()
selection_identifiers = list(stored_identifiers)
for specobjid in picks["specobjid"]:
    if specobjid not in selection_identifiers:
        selection_identifiers.append(specobjid)
pd.DataFrame({"specobjid": selection_identifiers}).to_csv(SELECTION_FILE, index=False)

# 4. Reference galaxies: the stored selection plus every galaxy that already has a fit file (calibration, mock and
# pilot fits are not galaxies), so a pass that changes the picks can never drop a fitted or split galaxy
fitted_identifiers = []
for file_name in sorted(os.listdir(FITS_DIRECTORY)):
    if file_name.endswith(".h5") and not file_name.startswith(("calib_", "mock_", "pilot")):
        fitted_identifiers.append(file_name[:-len(".h5")])
reference_identifiers = list(selection_identifiers)
for specobjid in fitted_identifiers:
    if specobjid not in reference_identifiers:
        reference_identifiers.append(specobjid)

available = set(matched["specobjid"])
lost_identifiers = [specobjid for specobjid in reference_identifiers if specobjid not in available]
if lost_identifiers:
    print(f"WARNING: {len(lost_identifiers)} stored galaxies are no longer in the S-PLUS x Firefly match "
          f"and cannot be rebuilt: {lost_identifiers}")
reference_identifiers = [specobjid for specobjid in reference_identifiers if specobjid in available]
selected = matched.set_index("specobjid").loc[reference_identifiers].reset_index()
selected = selected.rename(columns={"RA": "ra", "DEC": "dec", "Z_NOQSO": "z", "PLATE": "plate", "MJD": "mjd",
                                    "FIBERID": "fiberid", "Chabrier_MILES_age_massW": "firefly_age",
                                    "Chabrier_MILES_metallicity_massW": "firefly_metallicity",
                                    "Chabrier_MILES_total_mass": "firefly_mass"})
print(f"{len(selected)} reference galaxies: {len(stored_identifiers)} stored, "
      f"{len(selection_identifiers) - len(stored_identifiers)} newly picked, "
      f"{len(fitted_identifiers)} with a fit file")

## 2. Spectra
SDSS spectra are **not** corrected for Galactic extinction, so each is dereddened with the same map and law as the S-PLUS photometry (CSFD, CCM89, $R_V = 3.1$), converted to air wavelengths (the models' frame), masked (bad pixels, 5577/6300 Å sky lines) and rebinned to 5 Å.

In [ ]:
dust_map = CSFDQuery(map_fname=DUST_MAP_FILE, mask_fname=DUST_MASK_FILE)
galaxy_coordinates = SkyCoord(selected["ra"].to_numpy() * u.deg, selected["dec"].to_numpy() * u.deg)
# The map is a big-endian FITS image; float64 is native, which pandas needs
selected["ebv"] = dust_map(galaxy_coordinates).astype(float)

prepared_spectra = {}
for row in selected.itertuples():
    spectrum_file = os.path.join(SPECTRA_DIRECTORY, f"{row.specobjid}.fits")

    # 1. Download once
    if not os.path.exists(spectrum_file):
        try:
            downloaded = SDSS.get_spectra(plate=int(row.plate), mjd=int(row.mjd), fiberID=int(row.fiberid),
                                          data_release=16)
            downloaded[0].writeto(spectrum_file)
        except Exception as error:
            failures.append({"job_id": str(row.specobjid), "stage": "download", "error": repr(error)})
            continue

    # 2. Prepare for the fit
    with fits.open(spectrum_file) as spectrum_hdus:
        table = spectrum_hdus[1].data
        wavelength_vacuum = 10 ** table["loglam"].astype(float)
        flux = table["flux"].astype(float)
        ivar = table["ivar"].astype(float)
    try:
        prepared_spectra[row.specobjid] = sm.prepare_spectrum(wavelength_vacuum, flux, ivar, row.ebv)
    except ValueError as error:
        failures.append({"job_id": str(row.specobjid), "stage": "prepare", "error": repr(error)})

print(f"{len(prepared_spectra)} spectra ready; {len(failures)} failures so far")

## 3. Aperture photometry
3″-diameter S-PLUS magnitudes, matching the SDSS fibre, dereddened like `dr3_corrected.csv`. They set the mass normalisation of the labels and feed the coverage test; they never enter the spectral fit.

The raw (not dereddened) magnitudes of every S-PLUS x Firefly match are cached in `splus_aperture_photometry.csv`; the S-PLUS Cloud is queried only for selected galaxies missing from that cache (each query is retried up to `SPLUS_QUERY_ATTEMPTS` times), and every newly queried galaxy is appended to the cache, so a re-run needs no live queries.

In [ ]:
# Output column names, built once
aperture_columns = []
aperture_error_columns = []
for band in sm.BANDS:
    aperture_columns.append(f"{band}_aper")
    aperture_error_columns.append(f"e_{band}_aper")

# Columns of the cache file, in file order
cache_columns = ["specobjid", "ra", "dec", "splus_ra", "splus_dec", "match_separation_arcsec"]
for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
    cache_columns.append(magnitude_column)
    cache_columns.append(error_column)


def query_to_dataframe(result):
    """S-PLUS Cloud query result as a DataFrame.

    Inputs: result: what splusdata's Core.query returns, an astropy Table or a DataFrame.
    Returns: pandas DataFrame with the same rows and columns.
    """
    if hasattr(result, "to_pandas"):
        return result.to_pandas()
    return pd.DataFrame(result)


def cone_query(table, columns, ra, dec, radius_arcsec):
    """ADQL selecting columns of every object within radius of one position.

    One position per query: the S-PLUS Cloud rejects several OR-ed cones with "Column 'ra' does not exist".
    Inputs: table: S-PLUS Cloud table name (str); columns: names of the columns to select besides the
            position (list of str); ra, dec: cone centre [deg]; radius_arcsec: cone radius [arcsec].
    Returns: the ADQL query text (str).
    """
    radius_degrees = radius_arcsec / 3600
    column_list = ", ".join([SPLUS_RA_COLUMN, SPLUS_DEC_COLUMN, *columns])
    return (f"SELECT {column_list} FROM {table} WHERE 1 = CONTAINS(POINT('ICRS', {SPLUS_RA_COLUMN}, "
            f"{SPLUS_DEC_COLUMN}), CIRCLE('ICRS', {ra}, {dec}, {radius_degrees}))")


def run_query_with_retries(connection, query):
    """Run one S-PLUS Cloud query, trying again after a failure (the server is slow and sometimes drops requests).

    Waits SPLUS_RETRY_WAIT seconds after the first failure and twice as long after each further one.
    Inputs: connection: logged-in splusdata.Core; query: ADQL text (str).
    Returns: DataFrame with the query result (empty if no object was found).
    Raises: the last error if all SPLUS_QUERY_ATTEMPTS tries fail.
    """
    wait = SPLUS_RETRY_WAIT
    for attempt in range(1, SPLUS_QUERY_ATTEMPTS + 1):
        try:
            return query_to_dataframe(connection.query(query, execution_mode=SPLUS_EXECUTION_MODE,
                                                       timeout=SPLUS_QUERY_TIMEOUT))
        except Exception as error:
            if attempt == SPLUS_QUERY_ATTEMPTS:
                raise
            print(f"S-PLUS query failed (try {attempt} of {SPLUS_QUERY_ATTEMPTS}): {error!r}; "
                  f"trying again in {wait} s")
            time.sleep(wait)
            wait = 2 * wait


def query_one_galaxy(connection, specobjid, ra, dec):
    """Raw 3" aperture photometry of one galaxy, as one row of the cache file.

    Each band is a separate table, so there is one query per band; in each, the closest object within
    MATCH_RADIUS is taken. splus_ra, splus_dec and match_separation_arcsec describe the closest object of
    the first band in which one was found.
    Inputs: connection: logged-in splusdata.Core; specobjid: SDSS identifier (str); ra, dec: SDSS position [deg].
    Returns: dict with the cache_columns (magnitudes [mag] and errors [mag] are NaN where nothing was found).
    """
    row = dict.fromkeys(cache_columns, np.nan)
    row.update(specobjid=specobjid, ra=ra, dec=dec)
    galaxy_position = SkyCoord(ra * u.deg, dec * u.deg)
    for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
        query = cone_query(table, [magnitude_column, error_column], ra, dec, MATCH_RADIUS)
        result = run_query_with_retries(connection, query)
        if len(result) == 0:
            continue
        result_positions = SkyCoord(result[SPLUS_RA_COLUMN].to_numpy() * u.deg,
                                    result[SPLUS_DEC_COLUMN].to_numpy() * u.deg)
        separations = result_positions.separation(galaxy_position).arcsec
        closest = np.argmin(separations)
        row[magnitude_column] = result[magnitude_column].iloc[closest]
        row[error_column] = result[error_column].iloc[closest]
        if np.isnan(row["splus_ra"]):
            row["splus_ra"] = result[SPLUS_RA_COLUMN].iloc[closest]
            row["splus_dec"] = result[SPLUS_DEC_COLUMN].iloc[closest]
            row["match_separation_arcsec"] = separations[closest]
    return row


def query_aperture_photometry(galaxies):
    """Raw 3" aperture magnitudes and errors of galaxies from the S-PLUS Cloud, saved to the cache as they arrive.

    Each galaxy is appended to APERTURE_CACHE_FILE as soon as it is queried (a galaxy with no S-PLUS object
    is stored as NaN), so an interrupted run loses nothing.
    Inputs: galaxies: DataFrame with columns specobjid (str), ra and dec [deg].
    Returns: DataFrame indexed like galaxies, with {band}_aper and e_{band}_aper columns [mag] (NaN if not found).
    """
    import splusdata
    from dotenv import load_dotenv

    load_dotenv(CREDENTIALS_FILE)
    connection = splusdata.Core(os.environ["SPLUS_USERNAME"], os.environ["SPLUS_PASSWORD"])

    photometry = pd.DataFrame(np.nan, index=galaxies.index, columns=aperture_columns + aperture_error_columns)
    for galaxy in galaxies.itertuples():
        row = query_one_galaxy(connection, galaxy.specobjid, galaxy.ra, galaxy.dec)
        write_header = not os.path.exists(APERTURE_CACHE_FILE)
        pd.DataFrame([row], columns=cache_columns).to_csv(APERTURE_CACHE_FILE, mode="a", header=write_header,
                                                          index=False)
        for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
            photometry.loc[galaxy.Index, f"{band}_aper"] = row[magnitude_column]
            photometry.loc[galaxy.Index, f"e_{band}_aper"] = row[error_column]
    return photometry


# 1. Raw magnitudes from the cache, by specobjid
for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
    selected[f"{band}_aper"] = np.nan
    selected[f"e_{band}_aper"] = np.nan
in_cache = pd.Series(False, index=selected.index)
if os.path.exists(APERTURE_CACHE_FILE):
    cache = pd.read_csv(APERTURE_CACHE_FILE, dtype={"specobjid": str})
    cache = cache.drop_duplicates("specobjid").set_index("specobjid")
    in_cache = selected["specobjid"].isin(cache.index)
    cached = cache.loc[selected.loc[in_cache, "specobjid"]]
    for band, (table, magnitude_column, error_column) in APERTURE_SOURCES.items():
        selected.loc[in_cache, f"{band}_aper"] = cached[magnitude_column].to_numpy()
        selected.loc[in_cache, f"e_{band}_aper"] = cached[error_column].to_numpy()
print(f"{in_cache.sum()} of {len(selected)} galaxies found in {APERTURE_CACHE_FILE}")

# 2. The rest from the S-PLUS Cloud (appended to the cache as they arrive)
if not in_cache.all():
    queried = query_aperture_photometry(selected[~in_cache])
    selected.loc[queried.index, queried.columns] = queried

# 3. Galactic extinction, as for dr3_corrected.csv (non-detections keep their 99)
raw_magnitudes = selected[aperture_columns].to_numpy()
dereddened = sm.deredden_magnitudes(raw_magnitudes, selected["ebv"].to_numpy())
measured = raw_magnitudes < sm.UNDETECTED_MAGNITUDE
selected[aperture_columns] = np.where(measured, dereddened, raw_magnitudes)

missing_photometry = selected["r_aper"].isna()
print(f"Aperture photometry found for {(~missing_photometry).sum()} of {len(selected)} galaxies")
assert (~missing_photometry).sum() >= 0.9 * len(selected), "Aperture photometry missing for >10% of galaxies"

## 4. Fits (outside the notebook)
MultiNest fits of real galaxies need hours each, so they run outside the notebook: this section writes one job file per fit to `spectral_fits/jobs/` and `run_spectral_fits.py` runs them, one single-core process per fit (see its docstring for the environment and the Slurm example `slurm_fit.sh`). Finished fits are `spectral_fits/<job_id>.h5`; copy them back and re-run from the calibration test on.

**Jobs:** every selected galaxy, plus `sm.N_CALIBRATION_MOCKS` calibration mocks drawn from the full training prior (`sfh_type` included), each with the wavelengths, mask and errors of a random selected galaxy. The mocks and their true labels are stored in `MOCK_FILE` and reused, so their fits stay checkable. Results of the first run (fits with a lower iteration cap than `sm.MAX_ITERATIONS`, and its mock fits) are first moved to `spectral_fits/superseded/`.

In [ ]:
def move_to_superseded(name):
    """Move a file or folder of FITS_DIRECTORY into SUPERSEDED_DIRECTORY, unless that name is already there.

    Inputs: name: file or folder name inside FITS_DIRECTORY (str).
    Returns: True if it was moved, False if it does not exist or the destination is taken (bool).
    """
    source = os.path.join(FITS_DIRECTORY, name)
    destination = os.path.join(SUPERSEDED_DIRECTORY, name)
    if not os.path.exists(source):
        return False
    if os.path.exists(destination):
        print(f"WARNING: {destination} already exists; {source} was left where it is")
        return False
    os.makedirs(SUPERSEDED_DIRECTORY, exist_ok=True)
    shutil.move(source, destination)
    return True


# 1. The first run's mock fits and their folders (names mock_<k>)
moved = []
for name in sorted(os.listdir(FITS_DIRECTORY)):
    if name.startswith("mock_") and move_to_superseded(name):
        moved.append(name)

# 2. Fits stored with a lower iteration cap than sm.MAX_ITERATIONS (or none), with their MultiNest folders.
# An unreadable file stays: the external run fits that job again
for name in sorted(os.listdir(FITS_DIRECTORY)):
    if not name.endswith(".h5"):
        continue
    try:
        _, metadata = sm.load_fit(os.path.join(FITS_DIRECTORY, name))
    except Exception:
        continue
    if metadata.get("max_iterations", 0) < sm.MAX_ITERATIONS:
        for moved_name in (name, name[:-len(".h5")]):
            if move_to_superseded(moved_name):
                moved.append(moved_name)

# 3. The first run's mock file; the calibration mocks are drawn afresh into MOCK_FILE
old_mock_name = os.path.basename(OLD_MOCK_FILE)
if move_to_superseded(old_mock_name):
    moved.append(old_mock_name)

print(f"Moved {len(moved)} items to {SUPERSEDED_DIRECTORY}: {moved}" if moved
      else f"Nothing to supersede: no earlier fit has a lower iteration cap than {sm.MAX_ITERATIONS}")

In [ ]:
def real_galaxy_job(specobjid):
    """Fit job of one selected galaxy.

    Inputs: specobjid: SDSS identifier (str) of a galaxy in selected with a prepared spectrum.
    Returns: dict with job_id (str), spectrum (dict from prepare_spectrum: wavelength [Angstrom], flux, ivar)
             and redshift (spectroscopic, dimensionless).
    """
    redshift = float(selected.loc[selected["specobjid"] == specobjid, "z"].iloc[0])
    return {"job_id": str(specobjid), "spectrum": prepared_spectra[specobjid], "redshift": redshift}


real_jobs = [real_galaxy_job(specobjid) for specobjid in prepared_spectra]
assert len(real_jobs) > 0, "No spectrum was prepared; there is nothing to fit"

# Calibration mocks: drawn once and stored with their true labels, because their fits are cached on disk and
# must be compared with the mocks they were made from. Delete MOCK_FILE (and the calib_* fits) to draw new ones
if os.path.exists(MOCK_FILE):
    calibration_jobs, calibration_truths = joblib.load(MOCK_FILE)
    print(f"Reusing {len(calibration_jobs)} stored calibration mocks from {MOCK_FILE}")
else:
    templates = [(job["spectrum"], job["redshift"]) for job in real_jobs]
    calibration_jobs, calibration_truths = sm.draw_calibration_mocks(
        sm.N_CALIBRATION_MOCKS, templates, np.random.default_rng(SEED))
    joblib.dump((calibration_jobs, calibration_truths), MOCK_FILE)
    print(f"Drew {len(calibration_jobs)} calibration mocks from the training prior, stored in {MOCK_FILE}")

In [ ]:
# 1. One job file per calibration mock and per selected galaxy
for job in [*calibration_jobs, *real_jobs]:
    sm.export_job(job, JOBS_DIRECTORY)
pd.DataFrame(failures).to_csv(FAILURES_FILE, index=False)
n_jobs = len(calibration_jobs) + len(real_jobs)
print(f"Exported {len(calibration_jobs)} calibration mocks and {len(real_jobs)} galaxies to {JOBS_DIRECTORY}/ "
      f"({len(failures)} download or prepare failures in {FAILURES_FILE})")

# 2. How to run them
run_command = f"python run_spectral_fits.py --jobs-dir {JOBS_DIRECTORY} --output-dir {FITS_DIRECTORY}"
print("\nOn a many-core PC, one fit per core (--workers defaults to the usable cores minus one):")
print(f"  LD_LIBRARY_PATH={LIBMULTINEST_DIRECTORY} {run_command} --workers <N>")
print(f"On a Slurm cluster: copy the bundle (see run_spectral_fits.py), set --array=0-{n_jobs - 1} in slurm_fit.sh, "
      "then  sbatch slurm_fit.sh")
print(f"Progress at any time:\n  LD_LIBRARY_PATH={LIBMULTINEST_DIRECTORY} {run_command} --status")
print(f"Then copy the <job_id>.h5 files back into {FITS_DIRECTORY}/ and re-run from the calibration test.")

# 3. Optional: run the fits here. The output is passed on line by line, so progress shows in the notebook
if RUN_FITS_HERE:
    print(f"\nRunning the fits here on {N_WORKERS} workers; interrupt the kernel to stop (finished fits are kept)")
    environment = dict(os.environ, LD_LIBRARY_PATH=LIBMULTINEST_DIRECTORY)
    process = subprocess.Popen([sys.executable, "run_spectral_fits.py", "--jobs-dir", JOBS_DIRECTORY,
                                "--output-dir", FITS_DIRECTORY, "--workers", str(N_WORKERS)],
                               env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end="")
    finally:
        process.terminate()
        process.wait()
    assert process.returncode == 0, "run_spectral_fits.py failed; see its output above"

## 5. Calibration test
Each calibration mock has known labels. The fitter is **calibrated** when, for `mw_age`, `Z_fin` and `f_old`, at least `sm.MIN_COVERAGE_95` (85%) of the true values lie inside the 95% interval of their posterior. The 68% coverage is reported too. The verdict goes into every row of the reference CSV as `fitter_calibrated` (empty until all mocks are fitted); it does not stop the notebook. A failure is a finding about the prior or the model, not a crash.

In [ ]:
def converged_from_multinest(job_id, metadata):
    """Whether a fit certainly stopped before its iteration limit, judged from its MultiNest folder.

    Uses the upper bound of the outer-loop iteration count (sm.multinest_iteration_bounds), so a capped
    run is never called converged. The limit is the one stored in the fit's metadata. A fit saved before
    the limit was stored (or without its MultiNest folder) keeps the flag stored when it ran: that flag
    came from the ev.dat row count, which overcounts iterations, so a stored True is already safe and a
    stored False is never upgraded against a limit that may differ from the one the fit used.
    Inputs: job_id: fit identifier, the name of its .h5 file and MultiNest folder (str); metadata: dict from sm.load_fit.
    Returns: True if the run converged, else False (bool).
    """
    iteration_limit = metadata.get("max_iterations")
    multinest_basename = os.path.join(FITS_DIRECTORY, job_id) + "/"
    if iteration_limit is None or not os.path.exists(multinest_basename + "ev.dat"):
        return bool(metadata["converged"])
    return sm.multinest_iteration_bounds(multinest_basename)[1] < iteration_limit


# 1. Posteriors of the calibration mocks that have finished
calibration_posteriors = {}
calibration_converged = 0
for job_id in calibration_truths:
    fit_file = os.path.join(FITS_DIRECTORY, f"{job_id}.h5")
    if os.path.exists(fit_file):
        calibration_posteriors[job_id], metadata = sm.load_fit(fit_file)
        calibration_converged += int(converged_from_multinest(job_id, metadata))

# 2. Coverage per label and interval
n_finished = len(calibration_posteriors)
n_expected = len(calibration_truths)
calibration_table = sm.calibration_coverage(calibration_posteriors, calibration_truths, sm.CALIBRATION_LABELS)
display(calibration_table.pivot(index="label", columns="level", values=["coverage", "n_jobs"]).round(2))

# 3. Verdict: only a complete set of mocks decides
coverage_95 = calibration_table[calibration_table["level"] == 0.95].set_index("label")["coverage"]
if n_finished < n_expected:
    fitter_calibrated = pd.NA
    print(f"INCOMPLETE: {n_finished} of {n_expected} calibration mocks fitted ({calibration_converged} converged); "
          "fitter_calibrated stays empty until all are done")
else:
    fitter_calibrated = bool((coverage_95 >= sm.MIN_COVERAGE_95).all())
    verdict = "PASS" if fitter_calibrated else "FAIL"
    print(f"{verdict}: 95% coverage {coverage_95.round(2).to_dict()} against the required {sm.MIN_COVERAGE_95} "
          f"({n_finished} mocks, {calibration_converged} converged)")

## 6. Labels, Firefly check, coverage and split
Each posterior sample's mass-like labels are rescaled by its aperture $r$ offset ($-0.4\,\Delta m$ dex), then summarised as 16/50/84th percentiles. A fit is **Firefly-consistent** if mass-weighted age and metallicity agree within 0.3 dex; failures are flagged, not deleted. Coverage uses the aperture photometry at the spectroscopic redshift. The table has every galaxy of the stored selection (`selection.csv`) and every galaxy with a fit file, so a later pass never drops one. Fits may be missing, partly or entirely: the cells below run with whatever exists and print the counts. `converged` is recomputed from each fit's MultiNest folder (`sm.multinest_iterations` below the iteration limit), because the stored flag of older fits counted one ev.dat row per mode and iteration. Galaxies that converged, are covered and are Firefly-consistent go to `validation` or `test` by a hash of `specobjid`. Read the output with `dtype={"specobjid": str}` (eBOSS ids exceed int64).

In [ ]:
LABEL_COLUMNS = ["log_ssfr", "mw_metallicity", "veldisp", "noise_scale", "Z_fin", "Av_ism",
                 *sm.SUMMARY_LABEL_COLUMNS, *sm.BIN_LABEL_COLUMNS]

# 1. Posterior summaries per galaxy (selected holds the stored selection and every fitted galaxy, see cell 4)
failed_identifiers = {failure["job_id"] for failure in failures}   # stage prepare or download
# run_spectral_fits.py writes fit_log.csv (local runs) and fit_logs/<job_id>.csv (one per Slurm task). A line cut
# short by a copy is skipped, a repeated header row is dropped, an empty or unparseable file is ignored with a warning
task_logs = sorted(glob.glob(os.path.join(FITS_DIRECTORY, "fit_logs", "*.csv")))
log_files = [os.path.join(FITS_DIRECTORY, "fit_log.csv"), *task_logs]
log_tables = []
for log_file in log_files:
    if os.path.exists(log_file):
        try:
            log_tables.append(pd.read_csv(log_file, dtype=str, on_bad_lines="skip"))
        except (pd.errors.EmptyDataError, pd.errors.ParserError) as log_error:
            print(f"WARNING: {log_file} ignored ({log_error})")
fit_log = pd.concat(log_tables, ignore_index=True) if log_tables else pd.DataFrame(columns=["job_id", "status"])
if {"job_id", "status"} <= set(fit_log.columns):
    fit_log = fit_log[fit_log["job_id"] != "job_id"]
    failed_identifiers |= set(fit_log.loc[fit_log["status"] == "failed", "job_id"])
rows = []
for galaxy in selected.itertuples():
    fit_file = os.path.join(FITS_DIRECTORY, f"{galaxy.specobjid}.h5")
    row = {"specobjid": galaxy.specobjid}
    if not os.path.exists(fit_file):
        # Failed galaxies are told apart from those the budget left for a later pass
        row["fit_status"] = "failed" if galaxy.specobjid in failed_identifiers else "not_fitted"
        rows.append(row)
        continue
    posterior, metadata = sm.load_fit(fit_file)

    # Mass-like labels follow the aperture r magnitude: log X = log X_model - 0.4 (r_obs - r_model)
    offsets = galaxy.r_aper - posterior["model_r"]
    posterior = sm.rescale_mass_labels(posterior, offsets)
    posterior["log_ssfr"] = posterior["log_sfr"] - posterior["log_mstar"]   # normalisation-free

    row["fit_status"] = "fitted"
    row["converged"] = converged_from_multinest(galaxy.specobjid, metadata)
    row["fit_hours"] = metadata["run_time"] / 3600
    row.update(sm.summarise_posterior(posterior, LABEL_COLUMNS))
    rows.append(row)
reference = selected.merge(pd.DataFrame(rows), on="specobjid", how="left")
# With no fit yet (or only some) the summary columns may be missing: add them empty, so later steps run
empty_summary = pd.DataFrame(np.zeros((1, len(LABEL_COLUMNS))), columns=LABEL_COLUMNS)
expected_columns = ["converged", "fit_hours", *sm.summarise_posterior(empty_summary, LABEL_COLUMNS)]
missing_columns = [column for column in expected_columns if column not in reference.columns]
reference = reference.reindex(columns=[*reference.columns, *missing_columns])

# 2. Firefly check; its missing values (-9999) become NaN before any logarithm, as in sm.compare_with_firefly
for column in ("firefly_age", "firefly_metallicity", "firefly_mass"):
    reference[column] = reference[column].where(reference[column] > sm.FIREFLY_MISSING)
firefly_check = sm.compare_with_firefly(
    reference["mw_age_median"].to_numpy(), reference["mw_metallicity_median"].to_numpy(),
    reference["firefly_age"].to_numpy(), reference["firefly_metallicity"].to_numpy(),
)
for column in firefly_check.columns:
    reference[column] = firefly_check[column].to_numpy()
reference["delta_log_mass"] = reference["log_mstar_median"] - np.log10(reference["firefly_mass"])

# 3. Coverage of the aperture photometry at the spectroscopic redshift
training = pd.read_parquet(TRAINING_FILE, columns=["redshift", *sm.BANDS])
aperture_magnitudes = reference[aperture_columns].to_numpy()
aperture_errors = reference[aperture_error_columns].to_numpy()
aperture_detected = sm.detected_bands(aperture_magnitudes, aperture_errors)
coverage = sm.coverage_test(aperture_magnitudes, aperture_errors, aperture_detected,
                            reference["z"].to_numpy(), np.full(len(reference), sm.DZ_MIN),
                            training[sm.BANDS].to_numpy(), training["redshift"].to_numpy())
for column in coverage.columns:
    reference[column] = coverage[column].to_numpy()

# 4. Locked split
usable = (reference["converged"].eq(True)
          & reference["covered"].astype(bool)
          & reference["firefly_consistent"].eq(True))
reference["split"] = "excluded"
for row_index in np.flatnonzero(usable.to_numpy()):
    reference.loc[row_index, "split"] = sm.split_by_hash(reference.loc[row_index, "specobjid"])

# The calibration verdict of the fitter (cell above) is stored on every row; empty while the mocks are incomplete
reference["fitter_calibrated"] = fitter_calibrated

reference.to_csv(OUTPUT_FILE, index=False)
fitted = reference["fit_status"] == "fitted"
assert reference.loc[fitted, "mw_age_median"].notna().all(), "A fitted galaxy has no labels in the output"
print(f"{OUTPUT_FILE}: {len(reference)} galaxies, {fitted.sum()} fitted, "
      f"{reference['converged'].eq(True).sum()} converged, {reference['covered'].sum()} covered, "
      f"{reference['firefly_consistent'].eq(True).sum()} Firefly-consistent; "
      f"fitter_calibrated = {fitter_calibrated}")
display(reference["split"].value_counts())

In [ ]:
reference["fibre"] = reference["fiber_diameter"].map({3.0: "SDSS 3\"", 2.0: "BOSS 2\""})
reference["galaxy_type"] = np.where(reference["log_ssfr_median"] > SSFR_PASSIVE, "star-forming", "passive")

if not fitted.any():
    print("No fits yet: nothing to compare with Firefly")
else:
    for grouping in ("fibre", "galaxy_type"):
        grouped = reference[fitted].groupby(grouping)
        table = pd.DataFrame({
            "median Δlog age": grouped["delta_log_age"].median(),
            "scatter Δlog age": grouped["delta_log_age"].std(),
            "median Δlog Z": grouped["delta_log_metallicity"].median(),
            "scatter Δlog Z": grouped["delta_log_metallicity"].std(),
            "median Δlog M": grouped["delta_log_mass"].median(),
            "consistent": grouped["firefly_consistent"].mean(),
            "galaxies": grouped.size(),
        })
        display(table.round(3))

    figure, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    comparisons = [
        (np.log10(reference["firefly_age"]), np.log10(reference["mw_age_median"] * sm.GYR_TO_YEARS), "log age [yr]"),
        (np.log10(reference["firefly_metallicity"]), np.log10(reference["mw_metallicity_median"]), r"log $Z/Z_\odot$"),
    ]
    for axis, (firefly_values, bagpipes_values, label) in zip(axes, comparisons):
        for galaxy_type, marker in (("star-forming", "o"), ("passive", "s")):
            chosen = fitted & (reference["galaxy_type"] == galaxy_type)
            axis.scatter(firefly_values[chosen], bagpipes_values[chosen], marker=marker, label=galaxy_type)
        limits = [np.nanmin([firefly_values, bagpipes_values]), np.nanmax([firefly_values, bagpipes_values])]
        axis.plot(limits, limits, color="black", lw=1)
        axis.set(xlabel=f"Firefly {label}", ylabel=f"Bagpipes {label}")
        axis.legend()
    figure.tight_layout()
    figure.savefig(FIGURE_FILE, dpi=200)
    plt.show()

## Next pass
Raise `GALAXIES_PER_CELL` and re-run: the selection in `selection.csv` only grows (new picks are appended, every stored or fitted galaxy stays in the table), the new jobs are exported next to the old ones, and `run_spectral_fits.py` fits only the jobs without an `.h5` file. The stored calibration mocks (`MOCK_FILE`) are reused, so the calibration test stays valid, and the hash split never moves an existing galaxy. Delete `MOCK_FILE` and the `calib_*` fits to draw new mocks. Add DR4 Stripe 82 galaxies by pointing `SPLUS_FILE` at a DR4 catalogue with the same columns.